# Vježba 5: *Zadatak za samostalan rad*


> Drazen Baric

In [1]:
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

#### Priprema
> podaci

In [2]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.boston_housing.load_data()
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

    0/57026 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

16384/57026 ━━━━━━━━━━━━━━━━━━━━ 0s 4us/step

57026/57026 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step


#### Rješenje 1.
> `konstruiraj_model` s brojem neurona kao parametrima

In [3]:
def konstruiraj_model(n1=64, n2=64):
    model = tf.keras.models.Sequential([
        tf.keras.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(n1, activation='relu'),
        tf.keras.layers.Dense(n2, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    model.compile(loss='mse', optimizer='rmsprop', metrics=['mae'])
    return model

#### Rješenje 2.
> K-struka unakrsna validacija (K=5)

In [4]:
kf = KFold(n_splits=5)
arhitekture = [(64, 64), (64, 32)]
batch_sizes = [8, 16, 32, 64]
rezultati = {}

for arh in arhitekture:
    for bs in batch_sizes:
        tf.keras.utils.set_random_seed(7)
        maes = []
        for train, val in kf.split(X_train):
            tf.keras.backend.clear_session()
            model = konstruiraj_model(*arh)
            model.fit(X_train[train], y_train[train], epochs=50, batch_size=bs, verbose=0)
            maes.append(model.evaluate(X_train[val], y_train[val], verbose=0)[1])
        rezultati[(arh, bs)] = np.mean(maes)
        print('arhitektura {}, batch_size {}: MAE = {:.4f}'.format(arh, bs, rezultati[(arh, bs)]))

najbolja_arh, najbolji_bs = min(rezultati, key=rezultati.get)
print('\nnajbolja kombinacija: arhitektura {}, batch_size {}'.format(najbolja_arh, najbolji_bs))

arhitektura (64, 64), batch_size 8: MAE = 2.6783


arhitektura (64, 64), batch_size 16: MAE = 2.7102


arhitektura (64, 64), batch_size 32: MAE = 2.7435


arhitektura (64, 64), batch_size 64: MAE = 2.8571


arhitektura (64, 32), batch_size 8: MAE = 2.5932


arhitektura (64, 32), batch_size 16: MAE = 2.6139


arhitektura (64, 32), batch_size 32: MAE = 2.6861


arhitektura (64, 32), batch_size 64: MAE = 2.9903

najbolja kombinacija: arhitektura (64, 32), batch_size 8


#### Rješenje 3.
> treniranje na cijelom skupu za treniranje s najboljom kombinacijom

In [5]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(7)
model = konstruiraj_model(*najbolja_arh)
model.fit(X_train, y_train, epochs=50, batch_size=najbolji_bs, verbose=0)

#### Rješenje 4.
> evaluacija na skupu za testiranje i usporedba s fiksnim hiperparametrima (64, 64, `batch_size`=16)

In [6]:
test_mse, test_mae = model.evaluate(X_test, y_test, verbose=0)
print('odabrani hiperparametri: MSE = {:.4f}, MAE = {:.4f}'.format(test_mse, test_mae))

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(7)
baseline = konstruiraj_model(64, 64)
baseline.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)
b_mse, b_mae = baseline.evaluate(X_test, y_test, verbose=0)
print('fiksni hiperparametri:   MSE = {:.4f}, MAE = {:.4f}'.format(b_mse, b_mae))

odabrani hiperparametri: MSE = 20.8466, MAE = 2.7818


fiksni hiperparametri:   MSE = 23.3355, MAE = 3.0337
